In [17]:
from langchain_community.llms import Ollama
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

In [18]:
llm = Ollama(model="llama2:7b")

In [19]:
loader = CSVLoader(file_path="../data/magic_loops_data.csv")

data_csv = loader.load()

In [20]:
for document in loader.lazy_load():
    print(document)

page_content='id: 1
temperature: 74.91446090291294
timestamp: 2025-01-16 23:51:20' metadata={'source': '../data/magic_loops_data.csv', 'row': 0}
page_content='id: 1
temperature: 50.721744450120774
timestamp: 2025-01-16 23:51:20' metadata={'source': '../data/magic_loops_data.csv', 'row': 1}
page_content='id: 1
temperature: 56.532609280492714
timestamp: 2025-01-16 23:51:20' metadata={'source': '../data/magic_loops_data.csv', 'row': 2}
page_content='id: 1
temperature: 93.00655275726456
timestamp: 2025-01-16 23:51:20' metadata={'source': '../data/magic_loops_data.csv', 'row': 3}
page_content='id: 1
temperature: 52.22153189502545
timestamp: 2025-01-16 23:51:20' metadata={'source': '../data/magic_loops_data.csv', 'row': 4}
page_content='id: 1
temperature: 62.80576538102713
timestamp: 2025-01-16 23:51:20' metadata={'source': '../data/magic_loops_data.csv', 'row': 5}
page_content='id: 1
temperature: 53.37814349990704
timestamp: 2025-01-16 23:51:20' metadata={'source': '../data/magic_loops_data

In [21]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=500)
docs = text_splitter.split_documents(data_csv)

In [22]:
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
embed_model = FastEmbedEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [23]:
vs = Chroma.from_documents(
    documents=docs,
    embedding=embed_model,
    persist_directory="../db/chroma_db_dir",  # Local mode with in-memory storage only
    collection_name="stanford_report_data"
)

In [ ]:
vectorstore = Chroma(embedding_function=embed_model,
                     persist_directory="../db/chroma_db_dir",
                     collection_name="stanford_report_data")
retriever=vectorstore.as_retriever(search_kwargs={'k': 3})

In [25]:
custom_prompt_template = """Usa la siguiente información para responder a la pregunta del usuario.
Si no sabes la respuesta, simplemente di que no lo sabes, no intentes inventar una respuesta.

Contexto: {context}
Pregunta: {question}

Solo devuelve la respuesta útil a continuación y nada más y responde siempre en español
Respuesta útil:
"""
prompt = PromptTemplate(template=custom_prompt_template,
                        input_variables=['context', 'question'])

In [26]:
qa = RetrievalQA.from_chain_type(llm=llm,
                                 chain_type="stuff",
                                 retriever=retriever,
                                 return_source_documents=True,
                                 chain_type_kwargs={"prompt": prompt})

In [34]:
response = qa.invoke({"query": "Dame el promedio de las temperaturas de la maquina del csv que te di se lo mas exacto posible"})

In [35]:
response['result']

'El promedio de las temperaturas de la máquina es de 77.65°C.'